In [1]:
# Block 1 — setup + read IPUMS XML schema + read raw .dat
import xml.etree.ElementTree as ET
from pyspark.sql import SparkSession, functions as F

#spark = SparkSession.builder.getOrCreate()

spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "18g") \
    .config("spark.executor.instances", 7) \
    .getOrCreate()

xml_path = "../../evargasnavarro/shared/processed/usa_00001.xml" #To Edwin: Change this
dat_path = "../../evargasnavarro/shared/processed/usa_00001.dat" #To Edwin: Change this

# Parse IPUMS DDI XML for fixed-width specs
ns = {"ddi": "ddi:codebook:2_5"}
root = ET.parse(xml_path).getroot()

specs = []
for var in root.findall(".//ddi:dataDscr/ddi:var", ns):
    loc = var.find("ddi:location", ns)
    fmt = var.find("ddi:varFormat", ns)
    specs.append({
        "name": var.attrib["name"],
        "start": int(loc.attrib["StartPos"]),
        "width": int(loc.attrib["width"]),
        "dcml": int(var.attrib.get("dcml", "0")),
        "type": fmt.attrib.get("type", "character") if fmt is not None else "character"
    })

raw_df = spark.read.text(dat_path)
raw_df.show(5, truncate=False)  # raw fixed-width lines
print(f"Columns in XML spec: {len(specs)}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [2]:
# Block 2 — parse to Spark DataFrame + cast numerics + inspect
df = raw_df.select(*[
    F.substring("value", s["start"], s["width"]).alias(s["name"])
    for s in specs
])

for s in specs:
    if s["type"] == "numeric":
        df = df.withColumn(s["name"], F.col(s["name"]).cast("double"))
        if s["dcml"] > 0:
            df = df.withColumn(s["name"], F.col(s["name"]) / (10 ** s["dcml"]))


df = df.select("YEAR", "STATEFIP", "SEX", "AGE", "RACE", "EDUC", "INCTOT", "CPI99")
df.show(10, truncate=False)

+------+--------+---+----+----+----+---------+-----+
|YEAR  |STATEFIP|SEX|AGE |RACE|EDUC|INCTOT   |CPI99|
+------+--------+---+----+----+----+---------+-----+
|2001.0|1.0     |1.0|39.0|1.0 |3.0 |6400.0   |0.941|
|2001.0|1.0     |1.0|33.0|2.0 |10.0|30000.0  |0.941|
|2001.0|1.0     |2.0|40.0|1.0 |6.0 |16500.0  |0.941|
|2001.0|1.0     |1.0|10.0|1.0 |1.0 |9999999.0|0.941|
|2001.0|1.0     |2.0|21.0|1.0 |3.0 |0.0      |0.941|
|2001.0|1.0     |1.0|39.0|1.0 |6.0 |21000.0  |0.941|
|2001.0|1.0     |2.0|55.0|1.0 |10.0|12000.0  |0.941|
|2001.0|1.0     |1.0|38.0|1.0 |3.0 |20000.0  |0.941|
|2001.0|1.0     |2.0|32.0|1.0 |2.0 |0.0      |0.941|
|2001.0|1.0     |1.0|10.0|1.0 |1.0 |9999999.0|0.941|
+------+--------+---+----+----+----+---------+-----+
only showing top 10 rows



# Complete Preprocessing using Spark

### Replacing code scheme defined missing values in INCTOT with Averages grouped by Year
Because of inflation, it's better to replace missing values with the average income of that year rather than the global average. These replaced values will be more representative of the respective year they would have been reported.

In [3]:
df = df.replace(9999999.0, None, subset=["INCTOT"]) #Exclude missing values from average calculations
df.show(10)

+------+--------+---+----+----+----+-------+-----+
|  YEAR|STATEFIP|SEX| AGE|RACE|EDUC| INCTOT|CPI99|
+------+--------+---+----+----+----+-------+-----+
|2001.0|     1.0|1.0|39.0| 1.0| 3.0| 6400.0|0.941|
|2001.0|     1.0|1.0|33.0| 2.0|10.0|30000.0|0.941|
|2001.0|     1.0|2.0|40.0| 1.0| 6.0|16500.0|0.941|
|2001.0|     1.0|1.0|10.0| 1.0| 1.0|   NULL|0.941|
|2001.0|     1.0|2.0|21.0| 1.0| 3.0|    0.0|0.941|
|2001.0|     1.0|1.0|39.0| 1.0| 6.0|21000.0|0.941|
|2001.0|     1.0|2.0|55.0| 1.0|10.0|12000.0|0.941|
|2001.0|     1.0|1.0|38.0| 1.0| 3.0|20000.0|0.941|
|2001.0|     1.0|2.0|32.0| 1.0| 2.0|    0.0|0.941|
|2001.0|     1.0|1.0|10.0| 1.0| 1.0|   NULL|0.941|
+------+--------+---+----+----+----+-------+-----+
only showing top 10 rows



In [4]:
income_averages = df.select("YEAR", "INCTOT").groupBy("YEAR").agg(F.avg("INCTOT").alias("INCTOT_AVG"))
income_averages = income_averages.select("YEAR", "INCTOT_AVG").rdd.collectAsMap()
income_averages #averages per year

{2001.0: 29386.547426435278,
 2002.0: 29917.722725334796,
 2003.0: 29942.784641597074,
 2004.0: 31199.262673265664,
 2005.0: 32081.598902228532,
 2006.0: 32496.915978168345,
 2007.0: 34351.79880858902,
 2008.0: 35465.87360155502,
 2009.0: 34661.97982040677,
 2010.0: 33779.009013067865,
 2011.0: 32768.56448544411,
 2012.0: 34117.01523478559,
 2013.0: 35747.96799509845,
 2014.0: 36528.664186469454,
 2015.0: 38276.89470220228,
 2016.0: 39487.482940526665,
 2017.0: 40832.207084073074,
 2018.0: 42496.599239466035,
 2019.0: 45302.28290449767,
 2020.0: 44912.61504463044,
 2021.0: 46295.92238451688,
 2022.0: 49422.73825908559,
 2023.0: 52133.932345992565,
 2024.0: 54654.70966942347}

In [5]:
for key, val in income_averages.items():
    df = df.withColumn("INCTOT", F.when((F.col("YEAR") == key) & (F.col("INCTOT").isNull()), val).otherwise(F.col("INCTOT")))
df.show(10)

+------+--------+---+----+----+----+------------------+-----+
|  YEAR|STATEFIP|SEX| AGE|RACE|EDUC|            INCTOT|CPI99|
+------+--------+---+----+----+----+------------------+-----+
|2001.0|     1.0|1.0|39.0| 1.0| 3.0|            6400.0|0.941|
|2001.0|     1.0|1.0|33.0| 2.0|10.0|           30000.0|0.941|
|2001.0|     1.0|2.0|40.0| 1.0| 6.0|           16500.0|0.941|
|2001.0|     1.0|1.0|10.0| 1.0| 1.0|29386.547426435278|0.941|
|2001.0|     1.0|2.0|21.0| 1.0| 3.0|               0.0|0.941|
|2001.0|     1.0|1.0|39.0| 1.0| 6.0|           21000.0|0.941|
|2001.0|     1.0|2.0|55.0| 1.0|10.0|           12000.0|0.941|
|2001.0|     1.0|1.0|38.0| 1.0| 3.0|           20000.0|0.941|
|2001.0|     1.0|2.0|32.0| 1.0| 2.0|               0.0|0.941|
|2001.0|     1.0|1.0|10.0| 1.0| 1.0|29386.547426435278|0.941|
+------+--------+---+----+----+----+------------------+-----+
only showing top 10 rows



### Adjusting for inflation
According to the IPUMS website, "Users studying change over time must adjust for inflation. Consumer Price Index adjustment factors for the appropriate years can be found in the CPI99 variable". Therefore, we will multiply the CPI99 column to adjust incomes for inflation so they are comparable across years.

In [6]:
df = df.withColumn("REALINCTOT", F.col("INCTOT") * F.col("CPI99"))
df = df.drop("CPI99") # No longer need CPI column
df.show(10)

+------+--------+---+----+----+----+------------------+------------------+
|  YEAR|STATEFIP|SEX| AGE|RACE|EDUC|            INCTOT|        REALINCTOT|
+------+--------+---+----+----+----+------------------+------------------+
|2001.0|     1.0|1.0|39.0| 1.0| 3.0|            6400.0|            6022.4|
|2001.0|     1.0|1.0|33.0| 2.0|10.0|           30000.0|           28230.0|
|2001.0|     1.0|2.0|40.0| 1.0| 6.0|           16500.0|           15526.5|
|2001.0|     1.0|1.0|10.0| 1.0| 1.0|29386.547426435278|27652.741128275597|
|2001.0|     1.0|2.0|21.0| 1.0| 3.0|               0.0|               0.0|
|2001.0|     1.0|1.0|39.0| 1.0| 6.0|           21000.0|           19761.0|
|2001.0|     1.0|2.0|55.0| 1.0|10.0|           12000.0|           11292.0|
|2001.0|     1.0|1.0|38.0| 1.0| 3.0|           20000.0|           18820.0|
|2001.0|     1.0|2.0|32.0| 1.0| 2.0|               0.0|               0.0|
|2001.0|     1.0|1.0|10.0| 1.0| 1.0|29386.547426435278|27652.741128275597|
+------+--------+---+----

### Mapping Categorical Mappings to New Columns
We will apply the mappings provided by IPUMS to create qualitiative columns

In [7]:
statefip = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    6: "California",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    28: "Mississippi",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    32: "Nevada",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    38: "North Dakota",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    50: "Vermont",
    51: "Virginia",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
    56: "Wyoming",
    72: "Puerto Rico",
    99: "State not identified"
}
sex = {
    1: "Male", 
    2: "Female", 
    9: "Missing/blank"
}
race = {
    1: "White", 
    2: "Black/African American", 
    3: "American Indian or Alaska Native", 
    4: "Chinese", 
    5: "Japanese", 
    6: "Other Asian or Pacific Islander", 
    7: "Other race, nec", 
    8: "Two major races", 
    9: "Three or more major races"
}
educ = {
    0: "N/A or no schooling", 
    1: "Nursery school to grade 4", 
    2: "Grade 5, 6, 7, or 8", 
    3: "Grade 9", 
    4: "Grade 10", 
    5: "Grade 11", 
    6: "Grade 12", 
    7: "1 year of college", 
    8: "2 years of college", 
    9: "3 years of college", 
    10: "4 years of college", 
    11: "5+ years of college", 
    99: "Missing"
}

In [8]:
# State
state_mapping = [F.lit(x) for item in statefip.items() for x in item]
df = df.withColumn("STATENAME", F.create_map(state_mapping)[F.col("STATEFIP")])

# Sex
sex_mapping = [F.lit(x) for item in sex.items() for x in item]
df = df.withColumn("SEXNAME", F.create_map(sex_mapping)[F.col("SEX")])

# Race
race_mapping = [F.lit(x) for item in race.items() for x in item]
df = df.withColumn("RACENAME", F.create_map(race_mapping)[F.col("RACE")])

# Educ
educ_mapping = [F.lit(x) for item in educ.items() for x in item]
df = df.withColumn("EDUCNAME", F.create_map(educ_mapping)[F.col("EDUC")])

df.show(10)

+------+--------+---+----+----+----+------------------+------------------+---------+-------+--------------------+--------------------+
|  YEAR|STATEFIP|SEX| AGE|RACE|EDUC|            INCTOT|        REALINCTOT|STATENAME|SEXNAME|            RACENAME|            EDUCNAME|
+------+--------+---+----+----+----+------------------+------------------+---------+-------+--------------------+--------------------+
|2001.0|     1.0|1.0|39.0| 1.0| 3.0|            6400.0|            6022.4|  Alabama|   Male|               White|             Grade 9|
|2001.0|     1.0|1.0|33.0| 2.0|10.0|           30000.0|           28230.0|  Alabama|   Male|Black/African Ame...|  4 years of college|
|2001.0|     1.0|2.0|40.0| 1.0| 6.0|           16500.0|           15526.5|  Alabama| Female|               White|            Grade 12|
|2001.0|     1.0|1.0|10.0| 1.0| 1.0|29386.547426435278|27652.741128275597|  Alabama|   Male|               White|Nursery school to...|
|2001.0|     1.0|2.0|21.0| 1.0| 3.0|               0.0|

In [9]:
# Quick check for any nulls aka missed values from any failed mappings. Ideally, everything is 0
all_null_counts = df.select([F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in ["STATENAME", "SEXNAME", "RACENAME", "EDUCNAME"]])
all_null_counts.show(truncate=False)

+---------+-------+--------+--------+
|STATENAME|SEXNAME|RACENAME|EDUCNAME|
+---------+-------+--------+--------+
|0        |0      |0       |0       |
+---------+-------+--------+--------+



### Encoding Categorical Data
We will One-Hot encode the data in the categorical columns "STATEFIP", "SEX", and "RACE" due to their unranked nature. Unfortunately, we cannot use the raw numbers as indexes because the code scheme has many unused intermediate numbers like 3 for "STATEFIP" which will lead to unused indexes and overly sparse arrays. We will instead use the newly created categorical columns to create a new index column and then conver these indexes to One-Hot encodings.

In [10]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

#OH for One-Hot
pairs = [("STATENAME", "STATE_INDEX", "STATE_OH"), 
         ("SEXNAME", "SEX_INDEX", "SEX_OH"), 
         ("RACENAME", "RACE_INDEX", "RACE_OH")]
steps = []
for in_col, mid_col, out_col in pairs:
    indexer = StringIndexer(inputCol=in_col, outputCol=mid_col)
    steps.append(indexer)
    encoder = OneHotEncoder(inputCol=mid_col, outputCol=out_col, dropLast=False)
    steps.append(encoder)
    
pipeline = Pipeline(stages=steps)
model = pipeline.fit(df)
df = model.transform(df)

df = df.drop("STATE_INDEX", "SEX_INDEX", "RACE_INDEX")
df.select("STATENAME", "STATE_OH", "SEXNAME", "SEX_OH", "RACENAME", "RACE_OH").show(10)

+---------+---------------+-------+-------------+--------------------+-------------+
|STATENAME|       STATE_OH|SEXNAME|       SEX_OH|            RACENAME|      RACE_OH|
+---------+---------------+-------+-------------+--------------------+-------------+
|  Alabama|(51,[22],[1.0])|   Male|(2,[1],[1.0])|               White|(9,[0],[1.0])|
|  Alabama|(51,[22],[1.0])|   Male|(2,[1],[1.0])|Black/African Ame...|(9,[1],[1.0])|
|  Alabama|(51,[22],[1.0])| Female|(2,[0],[1.0])|               White|(9,[0],[1.0])|
|  Alabama|(51,[22],[1.0])|   Male|(2,[1],[1.0])|               White|(9,[0],[1.0])|
|  Alabama|(51,[22],[1.0])| Female|(2,[0],[1.0])|               White|(9,[0],[1.0])|
|  Alabama|(51,[22],[1.0])|   Male|(2,[1],[1.0])|               White|(9,[0],[1.0])|
|  Alabama|(51,[22],[1.0])| Female|(2,[0],[1.0])|               White|(9,[0],[1.0])|
|  Alabama|(51,[22],[1.0])|   Male|(2,[1],[1.0])|               White|(9,[0],[1.0])|
|  Alabama|(51,[22],[1.0])| Female|(2,[0],[1.0])|               W

On the other hand, because the education levels in the "EDUC" column are ranked and nicely formatted, we can assign it ordinal encoding using the original indexes. We just need to remove index 9 (since it doesn't exist) by reducing the indexes 10 and 11 by 1

In [11]:
df.select("EDUC").show(10)

+----+
|EDUC|
+----+
| 3.0|
|10.0|
| 6.0|
| 1.0|
| 3.0|
| 6.0|
|10.0|
| 3.0|
| 2.0|
| 1.0|
+----+
only showing top 10 rows



In [12]:
df = df.withColumn("EDUC", F.when(F.col("EDUC") >= 10, F.col("EDUC") - 1).otherwise(F.col("EDUC")))
df.select("EDUC").show(10)

+----+
|EDUC|
+----+
| 3.0|
| 9.0|
| 6.0|
| 1.0|
| 3.0|
| 6.0|
| 9.0|
| 3.0|
| 2.0|
| 1.0|
+----+
only showing top 10 rows



### Z-score standardization
The continuous columns use widely different scalings. We will use Z-score standardization so each column has a mean of 0 and a standard deviation of 1. We use this over min-max normalization because it's not as susceptible to extreme values which we see in the REALINCTOT column (made abundantly clear by its high maximum and high std, yet low mean). 

In [13]:
from pyspark.ml.feature import StandardScaler, VectorAssembler

assembler = VectorAssembler(inputCols=["REALINCTOT"], outputCol="REALINCTOT_VEC")
df = assembler.transform(df)

scaler = StandardScaler(inputCol="REALINCTOT_VEC", outputCol="REALINCTOT_Z", withStd=True, withMean=True)
scalerModel = scaler.fit(df)
df = scalerModel.transform(df)

assembler = VectorAssembler(inputCols=["AGE"], outputCol="AGE_VEC")
df = assembler.transform(df)

scaler = StandardScaler(inputCol="AGE_VEC", outputCol="AGE_Z", withStd=True, withMean=True)
scalerModel = scaler.fit(df)
df = scalerModel.transform(df)

df = df.drop("AGE_VEC", "REALINCTOT_VEC")

df.select("REALINCTOT_Z", "AGE_Z").show(10)

+--------------------+--------------------+
|        REALINCTOT_Z|               AGE_Z|
+--------------------+--------------------+
|[-0.5817350346037...|[-0.0757780991651...|
|[0.02582573467254...|[-0.3302835024318...|
|[-0.3217196206337...|[-0.0333605319540...|
|[0.01003295003231...|[-1.3058875482872...|
|[-0.7464972771192...|[-0.8392943089651...|
|[-0.2058711688650...|[-0.0757780991651...|
|[-0.4375680724025...| [0.602902976212521]|
|[-0.2316152692580...|[-0.1181956663762...|
|[-0.7464972771192...|[-0.3727010696429...|
|[0.01003295003231...|[-1.3058875482872...|
+--------------------+--------------------+
only showing top 10 rows



### Final Preprocessed Dataset

In [14]:
df.show(10)

+------+--------+---+----+----+----+------------------+------------------+---------+-------+--------------------+--------------------+---------------+-------------+-------------+--------------------+--------------------+
|  YEAR|STATEFIP|SEX| AGE|RACE|EDUC|            INCTOT|        REALINCTOT|STATENAME|SEXNAME|            RACENAME|            EDUCNAME|       STATE_OH|       SEX_OH|      RACE_OH|        REALINCTOT_Z|               AGE_Z|
+------+--------+---+----+----+----+------------------+------------------+---------+-------+--------------------+--------------------+---------------+-------------+-------------+--------------------+--------------------+
|2001.0|     1.0|1.0|39.0| 1.0| 3.0|            6400.0|            6022.4|  Alabama|   Male|               White|             Grade 9|(51,[22],[1.0])|(2,[1],[1.0])|(9,[0],[1.0])|[-0.5817350346037...|[-0.0757780991651...|
|2001.0|     1.0|1.0|33.0| 2.0| 9.0|           30000.0|           28230.0|  Alabama|   Male|Black/African Ame...|  4

For the next person: The main preprocessed columns are REALINCTOT_Z, AGE_Z, EDUC, STATE_OH, SEX_OH, and RACE_OH. Most of the original columns are available for you to use as well as their named counterparts for the categorical columns (STATENAME, SEXNAME, RACENAME, EDUCNAME). Use them as necessary and let me know if a preprocessing step is missing or should be included.